In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.filters import should_filter_out_trip_due_to_abnormality
from mode_analysis_main.src.map_terminology import map_to_shared_mode_names
from mode_analysis_main.src.optimized_analysis import get_prediction_for_trip_rmove
from mode_analysis_main.src.calculate_speed import calculate_speed_for_dataframe, haversine

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_processed_5min_small_file.csv'
output_file_path = f'outputs/old/5min_small_file_database_metrics.csv'
filtered_trips_path = f'./outputs/5min_small_file_filtered_trips.csv'

input_df = pd.read_csv(input_path)

user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['traj_id'] != -99]

        trajectories_df['traj_id'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['traj_id'].astype(str)

        rows = []
        for traj_id, df in trajectories_df.groupby('traj_id'):

            df = df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
            df['timestamp'] = pd.to_datetime(df['timestamp'])
            df = df.sort_values('timestamp')

            # Calculate density
            number_of_records = len(df)
            if not df.empty:
                total_trip_time_minutes = (df['timestamp'].iloc[-1] - df['timestamp'].iloc[0]).total_seconds() / 60
                density_records_per_minute = number_of_records / total_trip_time_minutes if total_trip_time_minutes > 0 else 0.0
            else:
                total_trip_time_minutes = 0.0
                density_records_per_minute = 0.0

            total_distance_km = haversine(
                df['lat'].shift(1), df['lng'].shift(1),
                df['lat'], df['lng']
            ).sum()
            _, average_speed_kmh, max_speed_kmh = calculate_speed_for_dataframe(df, with_smoothing=False)

            if len(df) < 2:
                longest_time_gap_minutes = 0.0
            else:
                longest_time_gap_minutes = (df['timestamp'] - df['timestamp'].shift(1)).max().total_seconds() / 60

            trip_details = {
                'trip_id': traj_id,
                'number_of_records': number_of_records,
                'total_trip_time_minutes': round(total_trip_time_minutes, 3),
                'total_distance_km': round(total_distance_km, 3),
                'average_speed_kmh': round(average_speed_kmh, 3),
                'max_speed_kmh': round(max_speed_kmh, 3),
                'density_records_per_minute': round(density_records_per_minute, 3),
                'longest_time_gap_minutes': round(longest_time_gap_minutes, 3)
            }

            rows.append(trip_details)
            count += 1

        dataframe = pd.DataFrame(rows)
        dataframe = dataframe.set_index('trip_id')
        if user_count == 0:
            dataframe.to_csv(output_file_path, index=True, header=True, mode='w')
        else:
            dataframe.to_csv(output_file_path, index=True, header=False, mode='a')
    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
import pandas as pd

metrics_df = pd.read_csv('outputs/old/5min_small_file_database_metrics.csv')

print(metrics_df['longest_time_gap_minutes'].max())

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.filters import should_filter_out_trip_due_to_abnormality
from mode_analysis_main.src.map_terminology import map_to_shared_mode_names
from mode_analysis_main.src.optimized_analysis import get_prediction_for_trip_rmove
from mode_analysis_main.src.calculate_speed import calculate_speed_for_dataframe, haversine

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_processed_5min_small_file.csv'
output_file_path = f'./outputs/5min_small_file_cluster_metrics.csv'

input_df = pd.read_csv(input_path)

user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['cluster'] != -99]

        trajectories_df['cluster'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['cluster'].astype(str)

        rows = []
        for traj_id, df in trajectories_df.groupby('cluster'):

            df = df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
            df['timestamp'] = pd.to_datetime(df['timestamp'])
            df = df.sort_values('timestamp')

            # Calculate density
            number_of_records = len(df)
            if not df.empty:
                total_trip_time_minutes = (df['timestamp'].iloc[-1] - df['timestamp'].iloc[0]).total_seconds() / 60
                density_records_per_minute = number_of_records / total_trip_time_minutes if total_trip_time_minutes > 0 else 0.0
            else:
                total_trip_time_minutes = 0.0
                density_records_per_minute = 0.0

            total_distance_km = haversine(
                df['lat'].shift(1), df['lng'].shift(1),
                df['lat'], df['lng']
            ).sum()
            _, average_speed_kmh, max_speed_kmh = calculate_speed_for_dataframe(df, with_smoothing=False)

            if len(df) < 2:
                longest_time_gap_minutes = 0.0
            else:
                longest_time_gap_minutes = (df['timestamp'] - df['timestamp'].shift(1)).max().total_seconds() / 60

            trip_details = {
                'trip_id': traj_id,
                'number_of_records': number_of_records,
                'total_trip_time_minutes': round(total_trip_time_minutes, 3),
                'total_distance_km': round(total_distance_km, 3),
                'average_speed_kmh': round(average_speed_kmh, 3),
                'max_speed_kmh': round(max_speed_kmh, 3),
                'density_records_per_minute': round(density_records_per_minute, 3),
                'longest_time_gap_minutes': round(longest_time_gap_minutes, 3)
            }

            rows.append(trip_details)
            count += 1

        dataframe = pd.DataFrame(rows)
        dataframe = dataframe.set_index('trip_id')
        if user_count == 0:
            dataframe.to_csv(output_file_path, index=True, header=True, mode='w')
        else:
            dataframe.to_csv(output_file_path, index=True, header=False, mode='a')
    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.plot import plot_trip_on_map

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_processed_5min_small_file.csv'

input_df = pd.read_csv(input_path)

user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if user_count >= 5:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['traj_id'] != -99]

        trajectories_df['traj_id'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['traj_id'].astype(str)

        rows = []
        for traj_id, df in trajectories_df.groupby('traj_id'):
            points_dataframe = df
            points_dataframe = points_dataframe.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
            points_dataframe['timestamp'] = pd.to_datetime(points_dataframe['timestamp'])

            plot_trip_on_map(points_dataframe, include_transit=False)

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")


In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.plot import plot_trip_on_map

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_processed_5min_small_file.csv'

input_df = pd.read_csv(input_path)

user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if user_count >= 5:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['cluster'] != -99]

        trajectories_df['cluster'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['cluster'].astype(str)

        rows = []
        for traj_id, df in trajectories_df.groupby('cluster'):
            points_dataframe = df
            points_dataframe = points_dataframe.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
            points_dataframe['timestamp'] = pd.to_datetime(points_dataframe['timestamp'])

            plot_trip_on_map(points_dataframe, include_transit=False)

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.plot import plot_trip_on_map

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_processed_5min_100m_small_file.csv'

input_df = pd.read_csv(input_path)

user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if user_count >= 5:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['cluster'] != -99]

        trajectories_df['cluster'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['cluster'].astype(str)

        rows = []
        for traj_id, df in trajectories_df.groupby('cluster'):
            points_dataframe = df
            points_dataframe = points_dataframe.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
            points_dataframe['timestamp'] = pd.to_datetime(points_dataframe['timestamp'])

            plot_trip_on_map(points_dataframe, include_transit=False)

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_processed_15min_small_file.csv'

input_df = pd.read_csv(input_path)

user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if user_count >= 5:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['cluster'] != -99]

        trajectories_df['cluster'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['cluster'].astype(str)

        rows = []
        for cluster_id, df in trajectories_df.groupby('cluster'):
            points_dataframe = df
            points_dataframe = points_dataframe.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
            points_dataframe['timestamp'] = pd.to_datetime(points_dataframe['timestamp'])

            print(f'Cluster: {cluster_id}')

            distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                                  points_dataframe.iloc[0]['lng'],
                                                  points_dataframe.iloc[-1]['lat'],
                                                  points_dataframe.iloc[-1]['lng'])
            print(f'Distance covered (first to last): {distance_beginning_to_end}')

            bounds = (points_dataframe['lng'].min(),
                      points_dataframe['lat'].min(),
                      points_dataframe['lng'].max(),
                      points_dataframe['lat'].max())
            corner1 = (bounds[1], bounds[0])
            corner2 = (bounds[3], bounds[2])
            distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
            print(f'Max distance covered: {distance_across}')
            plot_trip_on_map(points_dataframe, include_transit=False)

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_processed_5min_100m_small_file.csv'

input_df = pd.read_csv(input_path)

user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if user_count >= 5:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['cluster'] != -99]

        trajectories_df['cluster'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['cluster'].astype(str)

        rows = []
        for cluster_id, df in trajectories_df.groupby('cluster'):
            points_dataframe = df
            points_dataframe = points_dataframe.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
            points_dataframe['timestamp'] = pd.to_datetime(points_dataframe['timestamp'])

            print(f'Cluster: {cluster_id}')

            distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                                  points_dataframe.iloc[0]['lng'],
                                                  points_dataframe.iloc[-1]['lat'],
                                                  points_dataframe.iloc[-1]['lng'])
            print(f'Distance covered (first to last): {distance_beginning_to_end}')

            bounds = (points_dataframe['lng'].min(),
                      points_dataframe['lat'].min(),
                      points_dataframe['lng'].max(),
                      points_dataframe['lat'].max())
            corner1 = (bounds[1], bounds[0])
            corner2 = (bounds[3], bounds[2])
            distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
            print(f'Max distance covered: {distance_across}')
            plot_trip_on_map(points_dataframe, include_transit=False)

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_processed_5min_100m_no_clustering.csv'

input_df = pd.read_csv(input_path)

processing_limit = 15
user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if total_trips_count >= processing_limit:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['cluster'] != -99]

        trajectories_df['cluster'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['cluster'].astype(str)

        rows = []
        for cluster_id, df in trajectories_df.groupby('cluster'):
            if count >= processing_limit:
                break

            points_dataframe = df
            points_dataframe = points_dataframe.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
            points_dataframe['timestamp'] = pd.to_datetime(points_dataframe['timestamp'])

            print(f'User: {user}, Cluster: {cluster_id}')
            print(f'Number of points: {len(points_dataframe)}')

            distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                                  points_dataframe.iloc[0]['lng'],
                                                  points_dataframe.iloc[-1]['lat'],
                                                  points_dataframe.iloc[-1]['lng'])
            print(f'Distance covered (first to last): {distance_beginning_to_end}')

            points_dataframe_minus_last = points_dataframe[:-1]
            bounds = (points_dataframe_minus_last['lng'].min(),
                      points_dataframe_minus_last['lat'].min(),
                      points_dataframe_minus_last['lng'].max(),
                      points_dataframe_minus_last['lat'].max())
            corner1 = (bounds[1], bounds[0])
            corner2 = (bounds[3], bounds[2])
            distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
            print(f'Max distance covered (excluding last): {distance_across}')

            ## ADD TIME GAP TO DEMONSTRATE WHETHER LONG GAPS CREATES STOPS OF ONE DATAPOINT WHENEVER TIME + DISTANCE ARE LARGE ENOUGH

            if len(df) < 2:
                longest_time_gap_minutes = 0.0
                final_time_gap = 0.0
            else:
                longest_time_gap_minutes = (points_dataframe['timestamp'] - points_dataframe['timestamp'].shift(1)).max().total_seconds() / 60
                final_time_gap = (points_dataframe.iloc[-1]['timestamp'] - points_dataframe.iloc[-2]['timestamp']).total_seconds() / 60
            print(f'Longest time gap: {longest_time_gap_minutes}')
            print(f'Time gap before end: {final_time_gap}')

            plot_trip_on_map(points_dataframe, include_transit=False)

            count += 1

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_processed_5min_100m_no_clustering.csv'

input_df = pd.read_csv(input_path)

processing_limit = 30
user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if total_trips_count >= processing_limit:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['traj_id'] != -99]

        # trajectories_df['traj_id'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['traj_id'].astype(str)

        rows = []
        for trip_id, df in trajectories_df.groupby('traj_id'):
            if count >= processing_limit:
                break

            points_dataframe = df
            points_dataframe = points_dataframe.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
            points_dataframe['timestamp'] = pd.to_datetime(points_dataframe['timestamp'])

            print(f'User: {user}, Trajectory: {trip_id}')
            print(f'Number of points: {len(points_dataframe)}')

            distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                                  points_dataframe.iloc[0]['lng'],
                                                  points_dataframe.iloc[-1]['lat'],
                                                  points_dataframe.iloc[-1]['lng'])
            print(f'Distance covered (first to last): {distance_beginning_to_end}')

            bounds = (points_dataframe['lng'].min(),
                      points_dataframe['lat'].min(),
                      points_dataframe['lng'].max(),
                      points_dataframe['lat'].max())
            corner1 = (bounds[1], bounds[0])
            corner2 = (bounds[3], bounds[2])
            distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
            print(f'Max distance covered: {distance_across}')

            ## ADD TIME GAP TO DEMONSTRATE WHETHER LONG GAPS CREATES STOPS OF ONE DATAPOINT WHENEVER TIME + DISTANCE ARE LARGE ENOUGH

            if len(df) < 2:
                longest_time_gap_minutes = 0.0
                final_time_gap = 0.0
            else:
                longest_time_gap_minutes = (points_dataframe['timestamp'] - points_dataframe['timestamp'].shift(1)).max().total_seconds() / 60
                final_time_gap = (points_dataframe.iloc[-1]['timestamp'] - points_dataframe.iloc[-2]['timestamp']).total_seconds() / 60
            print(f'Longest time gap: {longest_time_gap_minutes}')
            print(f'Time gap before end: {final_time_gap}')

            plot_trip_on_map(points_dataframe, include_transit=False)

            count += 1

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/072461a9ab5bfe12691e5cb3faf084560536ffe12e05e4caa3915091e275e65f_0_clusters.csv'

input_df = pd.read_csv(input_path)

try:
    points_dataframe = input_df
    points_dataframe = points_dataframe.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
    points_dataframe['timestamp'] = pd.to_datetime(points_dataframe['timestamp'])

    print(f'Processing file: {input_path}')
    print(f'Number of points: {len(points_dataframe)}')

    distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                          points_dataframe.iloc[0]['lng'],
                                          points_dataframe.iloc[-1]['lat'],
                                          points_dataframe.iloc[-1]['lng'])
    print(f'Distance covered (first to last): {distance_beginning_to_end}')

    points_dataframe_minus_last = points_dataframe[:-1]
    bounds = (points_dataframe_minus_last['lng'].min(),
              points_dataframe_minus_last['lat'].min(),
              points_dataframe_minus_last['lng'].max(),
              points_dataframe_minus_last['lat'].max())
    corner1 = (bounds[1], bounds[0])
    corner2 = (bounds[3], bounds[2])
    distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
    print(f'Max distance covered (excluding last): {distance_across}')

    ## ADD TIME GAP TO DEMONSTRATE WHETHER LONG GAPS CREATES STOPS OF ONE DATAPOINT WHENEVER TIME + DISTANCE ARE LARGE ENOUGH

    if len(df) < 2:
        longest_time_gap_minutes = 0.0
        final_time_gap = 0.0
    else:
        longest_time_gap_minutes = (points_dataframe['timestamp'] - points_dataframe['timestamp'].shift(1)).max().total_seconds() / 60
        final_time_gap = (points_dataframe.iloc[-1]['timestamp'] - points_dataframe.iloc[-2]['timestamp']).total_seconds() / 60
    print(f'Longest time gap: {longest_time_gap_minutes}')
    print(f'Time gap before end: {final_time_gap}')

    plot_trip_on_map(points_dataframe, include_transit=False)

    start_lat, start_lon, start_time = points_dataframe.iloc[0][['lat', 'lng', 'timestamp']]
    for index, row in points_dataframe.iterrows():
        distance = haversine(start_lat, start_lon, row['lat'], row['lng'])
        time_diff = (row['timestamp'] - start_time).total_seconds() / 60
        print(f'{index}: {distance} kilometers, {time_diff} minutes')

except Exception as e:
    print(f"Unexpected error processing user: {user}")
    print(f"Error: {e}")

print('Done!')
print(f'Rows processed: {len(input_df)}')

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

from collections import namedtuple
Trajectory = namedtuple('Trajectory', ['id', 'source', 'df'])
Cluster = namedtuple('Cluster', ['id', 'source', 'df'])

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_processed_5min_100m_no_clustering.csv'

input_df = pd.read_csv(input_path)
input_df = input_df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
input_df['timestamp'] = pd.to_datetime(input_df['timestamp'])

processing_limit = 30
user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if total_trips_count >= processing_limit:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['traj_id'] != -99]
        trajectory_dfs = [Trajectory(traj_id, 'Trajectory', df) for traj_id, df in trajectories_df.groupby('traj_id')]

        clusters_df = user_df[user_df['cluster'] != -99]
        cluster_dfs = [Cluster(cluster, 'Cluster', df) for cluster, df in clusters_df.groupby('cluster')]

        all_dfs = trajectory_dfs + cluster_dfs
        all_dfs_ordered = sorted(all_dfs, key=lambda tuple: tuple.df.iloc[0]['timestamp'])

        # trajectories_df['traj_id'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['traj_id'].astype(str)

        for tuple in all_dfs_ordered:
            if count >= processing_limit:
                break

            points_dataframe = tuple.df

            print(f'Processing {tuple.source} {tuple.id}')
            print(f'Starting point: ({points_dataframe.iloc[0]["lat"]}, {points_dataframe.iloc[0]["lng"]})')
            print(f'Ending point: ({points_dataframe.iloc[-1]["lat"]}, {points_dataframe.iloc[-1]["lng"]})')
            print(f'Number of points: {len(points_dataframe)}')

            distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                                  points_dataframe.iloc[0]['lng'],
                                                  points_dataframe.iloc[-1]['lat'],
                                                  points_dataframe.iloc[-1]['lng'])
            print(f'Distance covered (first to last): {distance_beginning_to_end}')

            points_dataframe_minus_last = points_dataframe[:-1]
            bounds = (points_dataframe_minus_last['lng'].min(),
                      points_dataframe_minus_last['lat'].min(),
                      points_dataframe_minus_last['lng'].max(),
                      points_dataframe_minus_last['lat'].max())
            corner1 = (bounds[1], bounds[0])
            corner2 = (bounds[3], bounds[2])
            distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
            print(f'Max distance covered (excluding last): {distance_across}')

            ## ADD TIME GAP TO DEMONSTRATE WHETHER LONG GAPS CREATES STOPS OF ONE DATAPOINT WHENEVER TIME + DISTANCE ARE LARGE ENOUGH

            if len(points_dataframe) < 2:
                longest_time_gap_minutes = 0.0
                final_time_gap = 0.0
            else:
                longest_time_gap_minutes = (points_dataframe['timestamp'] - points_dataframe['timestamp'].shift(1)).max().total_seconds() / 60
                final_time_gap = (points_dataframe.iloc[-1]['timestamp'] - points_dataframe.iloc[-2]['timestamp']).total_seconds() / 60
            print(f'Longest time gap: {longest_time_gap_minutes}')
            print(f'Time gap before end: {final_time_gap}')

            plot_trip_on_map(points_dataframe, include_transit=False)

            count += 1

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

from collections import namedtuple
Trajectory = namedtuple('Trajectory', ['id', 'source', 'df'])
Cluster = namedtuple('Cluster', ['id', 'source', 'df'])

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_processed_5min_100m_radius_100m_cluster.csv'

input_df = pd.read_csv(input_path, low_memory=False)
input_df = input_df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
input_df['timestamp'] = pd.to_datetime(input_df['timestamp'])

processing_limit = 30
user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if total_trips_count >= processing_limit:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['traj_id'] != -99]
        trajectory_dfs = [Trajectory(traj_id, 'Trajectory', df) for traj_id, df in trajectories_df.groupby('traj_id')]

        clusters_df = user_df[user_df['cluster'] != -99]
        cluster_dfs = [Cluster(cluster, 'Cluster', df) for cluster, df in clusters_df.groupby('cluster')]

        all_dfs = trajectory_dfs + cluster_dfs
        all_dfs_ordered = sorted(all_dfs, key=lambda tuple: tuple.df.iloc[0]['timestamp'])

        # trajectories_df['traj_id'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['traj_id'].astype(str)

        for tuple in all_dfs_ordered:
            if count >= processing_limit:
                break

            points_dataframe = tuple.df

            print(f'Processing {tuple.source} {tuple.id}')
            print(f'Starting point: ({points_dataframe.iloc[0]["lat"]}, {points_dataframe.iloc[0]["lng"]})')
            print(f'Ending point: ({points_dataframe.iloc[-1]["lat"]}, {points_dataframe.iloc[-1]["lng"]})')
            print(f'Number of points: {len(points_dataframe)}')

            distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                                  points_dataframe.iloc[0]['lng'],
                                                  points_dataframe.iloc[-1]['lat'],
                                                  points_dataframe.iloc[-1]['lng'])
            print(f'Distance covered (first to last): {distance_beginning_to_end}')

            points_dataframe_minus_last = points_dataframe[:-1]
            bounds = (points_dataframe_minus_last['lng'].min(),
                      points_dataframe_minus_last['lat'].min(),
                      points_dataframe_minus_last['lng'].max(),
                      points_dataframe_minus_last['lat'].max())
            corner1 = (bounds[1], bounds[0])
            corner2 = (bounds[3], bounds[2])
            distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
            print(f'Max distance covered (excluding last): {distance_across}')

            ## ADD TIME GAP TO DEMONSTRATE WHETHER LONG GAPS CREATES STOPS OF ONE DATAPOINT WHENEVER TIME + DISTANCE ARE LARGE ENOUGH

            if len(points_dataframe) < 2:
                longest_time_gap_minutes = 0.0
                final_time_gap = 0.0
            else:
                longest_time_gap_minutes = (points_dataframe['timestamp'] - points_dataframe['timestamp'].shift(1)).max().total_seconds() / 60
                final_time_gap = (points_dataframe.iloc[-1]['timestamp'] - points_dataframe.iloc[-2]['timestamp']).total_seconds() / 60
            print(f'Longest time gap: {longest_time_gap_minutes}')
            print(f'Time gap before end: {final_time_gap}')

            plot_trip_on_map(points_dataframe, include_transit=False)

            count += 1

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

from collections import namedtuple
Trajectory = namedtuple('Trajectory', ['id', 'source', 'df'])
Cluster = namedtuple('Cluster', ['id', 'source', 'df'])

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_processed_5min_optimized.csv'

input_df = pd.read_csv(input_path, low_memory=False)
input_df = input_df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
input_df['timestamp'] = pd.to_datetime(input_df['timestamp'])

processing_limit = 30
user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if total_trips_count >= processing_limit:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['traj_id'] != -99]
        trajectory_dfs = [Trajectory(traj_id, 'Trajectory', df) for traj_id, df in trajectories_df.groupby('traj_id')]

        clusters_df = user_df[user_df['cluster'] != -99]
        cluster_dfs = [Cluster(cluster, 'Cluster', df) for cluster, df in clusters_df.groupby('cluster')]

        all_dfs = trajectory_dfs + cluster_dfs
        all_dfs_ordered = sorted(all_dfs, key=lambda tuple: tuple.df.iloc[0]['timestamp'])

        # trajectories_df['traj_id'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['traj_id'].astype(str)

        for tuple in all_dfs_ordered:
            if count >= processing_limit:
                break

            points_dataframe = tuple.df

            print(f'Processing {tuple.source} {tuple.id}')
            print(f'Starting point: ({points_dataframe.iloc[0]["lat"]}, {points_dataframe.iloc[0]["lng"]})')
            print(f'Ending point: ({points_dataframe.iloc[-1]["lat"]}, {points_dataframe.iloc[-1]["lng"]})')
            print(f'Number of points: {len(points_dataframe)}')

            distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                                  points_dataframe.iloc[0]['lng'],
                                                  points_dataframe.iloc[-1]['lat'],
                                                  points_dataframe.iloc[-1]['lng'])
            print(f'Distance covered (first to last): {distance_beginning_to_end}')

            points_dataframe_minus_last = points_dataframe[:-1]
            bounds = (points_dataframe_minus_last['lng'].min(),
                      points_dataframe_minus_last['lat'].min(),
                      points_dataframe_minus_last['lng'].max(),
                      points_dataframe_minus_last['lat'].max())
            corner1 = (bounds[1], bounds[0])
            corner2 = (bounds[3], bounds[2])
            distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
            print(f'Max distance covered (excluding last): {distance_across}')

            ## ADD TIME GAP TO DEMONSTRATE WHETHER LONG GAPS CREATES STOPS OF ONE DATAPOINT WHENEVER TIME + DISTANCE ARE LARGE ENOUGH

            if len(points_dataframe) < 2:
                longest_time_gap_minutes = 0.0
                final_time_gap = 0.0
            else:
                longest_time_gap_minutes = (points_dataframe['timestamp'] - points_dataframe['timestamp'].shift(1)).max().total_seconds() / 60
                final_time_gap = (points_dataframe.iloc[-1]['timestamp'] - points_dataframe.iloc[-2]['timestamp']).total_seconds() / 60
            print(f'Longest time gap: {longest_time_gap_minutes}')
            print(f'Time gap before end: {final_time_gap}')

            plot_trip_on_map(points_dataframe, include_transit=False)

            count += 1

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

from collections import namedtuple
Trajectory = namedtuple('Trajectory', ['id', 'source', 'df'])
Cluster = namedtuple('Cluster', ['id', 'source', 'df'])

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/old/seattle_2000_processed_5min_small_file.csv'

input_df = pd.read_csv(input_path, low_memory=False)
input_df = input_df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
input_df['timestamp'] = pd.to_datetime(input_df['timestamp'])

processing_limit = 30
user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if total_trips_count >= processing_limit:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['traj_id'] != -99]
        trajectory_dfs = [Trajectory(traj_id, 'Trajectory', df) for traj_id, df in trajectories_df.groupby('traj_id')]

        clusters_df = user_df[user_df['cluster'] != -99]
        cluster_dfs = [Cluster(cluster, 'Cluster', df) for cluster, df in clusters_df.groupby('cluster')]

        all_dfs = trajectory_dfs + cluster_dfs
        all_dfs_ordered = sorted(all_dfs, key=lambda tuple: tuple.df.iloc[0]['timestamp'])

        # trajectories_df['traj_id'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['traj_id'].astype(str)

        for tuple in all_dfs_ordered:
            if count >= processing_limit:
                break

            points_dataframe = tuple.df

            print(f'Processing {tuple.source} {tuple.id}')
            print(f'Starting point: ({points_dataframe.iloc[0]["lat"]}, {points_dataframe.iloc[0]["lng"]})')
            print(f'Ending point: ({points_dataframe.iloc[-1]["lat"]}, {points_dataframe.iloc[-1]["lng"]})')
            print(f'Number of points: {len(points_dataframe)}')

            distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                                  points_dataframe.iloc[0]['lng'],
                                                  points_dataframe.iloc[-1]['lat'],
                                                  points_dataframe.iloc[-1]['lng'])
            print(f'Distance covered (first to last): {distance_beginning_to_end}')

            points_dataframe_minus_last = points_dataframe[:-1]
            bounds = (points_dataframe_minus_last['lng'].min(),
                      points_dataframe_minus_last['lat'].min(),
                      points_dataframe_minus_last['lng'].max(),
                      points_dataframe_minus_last['lat'].max())
            corner1 = (bounds[1], bounds[0])
            corner2 = (bounds[3], bounds[2])
            distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
            print(f'Max distance covered (excluding last): {distance_across}')

            ## ADD TIME GAP TO DEMONSTRATE WHETHER LONG GAPS CREATES STOPS OF ONE DATAPOINT WHENEVER TIME + DISTANCE ARE LARGE ENOUGH

            if len(points_dataframe) < 2:
                longest_time_gap_minutes = 0.0
                final_time_gap = 0.0
            else:
                longest_time_gap_minutes = (points_dataframe['timestamp'] - points_dataframe['timestamp'].shift(1)).max().total_seconds() / 60
                final_time_gap = (points_dataframe.iloc[-1]['timestamp'] - points_dataframe.iloc[-2]['timestamp']).total_seconds() / 60
            print(f'Longest time gap: {longest_time_gap_minutes}')
            print(f'Time gap before end: {final_time_gap}')

            plot_trip_on_map(points_dataframe, include_transit=False)

            count += 1

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

from collections import namedtuple
Trajectory = namedtuple('Trajectory', ['id', 'source', 'df'])
Cluster = namedtuple('Cluster', ['id', 'source', 'df'])

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_processed_5min_stop_200m_radius_small.csv'

input_df = pd.read_csv(input_path, low_memory=False)
input_df = input_df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
input_df['timestamp'] = pd.to_datetime(input_df['timestamp'])

processing_limit = 75
user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if total_trips_count >= processing_limit:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['traj_id'] != -99]
        trajectory_dfs = [Trajectory(traj_id, 'Trajectory', df) for traj_id, df in trajectories_df.groupby('traj_id')]

        clusters_df = user_df[user_df['cluster'] != -99]
        cluster_dfs = [Cluster(cluster, 'Cluster', df) for cluster, df in clusters_df.groupby('cluster')]

        all_dfs = trajectory_dfs + cluster_dfs
        all_dfs_ordered = sorted(all_dfs, key=lambda tuple: tuple.df.iloc[0]['timestamp'])

        # trajectories_df['traj_id'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['traj_id'].astype(str)

        for tuple in all_dfs_ordered:
            if count >= processing_limit:
                break

            points_dataframe = tuple.df

            print(f'Processing {tuple.source} {tuple.id}')
            print(f'Starting point: ({points_dataframe.iloc[0]["lat"]}, {points_dataframe.iloc[0]["lng"]})')
            print(f'Ending point: ({points_dataframe.iloc[-1]["lat"]}, {points_dataframe.iloc[-1]["lng"]})')
            print(f'Number of points: {len(points_dataframe)}')

            distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                                  points_dataframe.iloc[0]['lng'],
                                                  points_dataframe.iloc[-1]['lat'],
                                                  points_dataframe.iloc[-1]['lng'])
            print(f'Distance covered (first to last): {distance_beginning_to_end}')

            points_dataframe_minus_last = points_dataframe[:-1]
            bounds = (points_dataframe_minus_last['lng'].min(),
                      points_dataframe_minus_last['lat'].min(),
                      points_dataframe_minus_last['lng'].max(),
                      points_dataframe_minus_last['lat'].max())
            corner1 = (bounds[1], bounds[0])
            corner2 = (bounds[3], bounds[2])
            distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
            print(f'Max distance covered (excluding last): {distance_across}')

            ## ADD TIME GAP TO DEMONSTRATE WHETHER LONG GAPS CREATES STOPS OF ONE DATAPOINT WHENEVER TIME + DISTANCE ARE LARGE ENOUGH

            if len(points_dataframe) < 2:
                longest_time_gap_minutes = 0.0
                final_time_gap = 0.0
            else:
                longest_time_gap_minutes = (points_dataframe['timestamp'] - points_dataframe['timestamp'].shift(1)).max().total_seconds() / 60
                final_time_gap = (points_dataframe.iloc[-1]['timestamp'] - points_dataframe.iloc[-2]['timestamp']).total_seconds() / 60
            print(f'Longest time gap: {longest_time_gap_minutes}')
            print(f'Time gap before end: {final_time_gap}')

            plot_trip_on_map(points_dataframe, include_transit=False)

            count += 1

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from mode_analysis_main.src.calculate_speed import haversine
from mode_analysis_main.src.plot import plot_trip_on_map

from collections import namedtuple
Trajectory = namedtuple('Trajectory', ['id', 'source', 'df'])
Cluster = namedtuple('Cluster', ['id', 'source', 'df'])

today = date.today().strftime("%Y%m%d")

input_path = f'./outputs/seattle_2000_processed_5min_stop_100m_radius_small.csv'

input_df = pd.read_csv(input_path, low_memory=False)
input_df = input_df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
input_df['timestamp'] = pd.to_datetime(input_df['timestamp'])

processing_limit = 75
user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for user, user_df in input_df.groupby('user_ID'):

    if total_trips_count >= processing_limit:
        break

    count = 0
    try:
        print(f'Processing user {user_count}')

        # Remove stop points
        trajectories_df = user_df[user_df['traj_id'] != -99]
        trajectory_dfs = [Trajectory(traj_id, 'Trajectory', df) for traj_id, df in trajectories_df.groupby('traj_id')]

        clusters_df = user_df[user_df['cluster'] != -99]
        cluster_dfs = [Cluster(cluster, 'Cluster', df) for cluster, df in clusters_df.groupby('cluster')]

        all_dfs = trajectory_dfs + cluster_dfs
        all_dfs_ordered = sorted(all_dfs, key=lambda tuple: tuple.df.iloc[0]['timestamp'])

        # trajectories_df['traj_id'] = trajectories_df['user_ID'].astype(str) + '_' + trajectories_df['traj_id'].astype(str)

        for tuple in all_dfs_ordered:
            if count >= processing_limit:
                break

            points_dataframe = tuple.df

            print(f'Processing {tuple.source} {tuple.id}')
            print(f'Starting point: ({points_dataframe.iloc[0]["lat"]}, {points_dataframe.iloc[0]["lng"]})')
            print(f'Ending point: ({points_dataframe.iloc[-1]["lat"]}, {points_dataframe.iloc[-1]["lng"]})')
            print(f'Number of points: {len(points_dataframe)}')

            distance_beginning_to_end = haversine(points_dataframe.iloc[0]['lat'],
                                                  points_dataframe.iloc[0]['lng'],
                                                  points_dataframe.iloc[-1]['lat'],
                                                  points_dataframe.iloc[-1]['lng'])
            print(f'Distance covered (first to last): {distance_beginning_to_end}')

            points_dataframe_minus_last = points_dataframe[:-1]
            bounds = (points_dataframe_minus_last['lng'].min(),
                      points_dataframe_minus_last['lat'].min(),
                      points_dataframe_minus_last['lng'].max(),
                      points_dataframe_minus_last['lat'].max())
            corner1 = (bounds[1], bounds[0])
            corner2 = (bounds[3], bounds[2])
            distance_across = haversine(corner1[0], corner1[1], corner2[0], corner2[1])
            print(f'Max distance covered (excluding last): {distance_across}')

            ## ADD TIME GAP TO DEMONSTRATE WHETHER LONG GAPS CREATES STOPS OF ONE DATAPOINT WHENEVER TIME + DISTANCE ARE LARGE ENOUGH

            if len(points_dataframe) < 2:
                longest_time_gap_minutes = 0.0
                final_time_gap = 0.0
            else:
                longest_time_gap_minutes = (points_dataframe['timestamp'] - points_dataframe['timestamp'].shift(1)).max().total_seconds() / 60
                final_time_gap = (points_dataframe.iloc[-1]['timestamp'] - points_dataframe.iloc[-2]['timestamp']).total_seconds() / 60
            print(f'Longest time gap: {longest_time_gap_minutes}')
            print(f'Time gap before end: {final_time_gap}')

            plot_trip_on_map(points_dataframe, include_transit=False)

            count += 1

    except Exception as e:
        print(f"Unexpected error processing user: {user}")
        print(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")